In [7]:


from pyspark.sql import SparkSession
import os as os

OIDC_TOKEN = os.environ["OIDC_TOKEN"]
spark = (SparkSession.builder.appName("test").remote("sc://spark-connector-0-driver-svc:15002")
# AWS 
.config("spark.sql.catalog.test_aws.scope", "openid")
.config("spark.sql.catalog.test_aws.header.X-Iceberg-Access-Delegation", "vended-credentials")
.config("spark.sql.catalog.test_aws", "org.apache.iceberg.spark.SparkCatalog")
.config("spark.sql.catalog.test_aws.type", "rest")
.config("spark.sql.catalog.test_aws.uri", "https://emryspolaris.click/api/catalog")
.config("spark.sql.catalog.test_aws.cache-enabled", "false")
.config("spark.sql.catalog.test_aws.warehouse", "aws")
.config("spark.sql.catalog.test_aws.token", OIDC_TOKEN)
.config("spark.sql.catalog.test_aws.oauth2-server-uri", "https://emrysoauth.click/realms/polaris/protocol/openid-connect/token")

).getOrCreate()

print("Finished initializing spark session")



Finished initializing spark session


![](https://media.licdn.com/dms/image/v2/D4E0BAQGumdWW8R2giQ/company-logo_200_200/B4EZ87y9pLK4AE-/0/1783414643504/health_data_research_service_logo?e=2147483647&v=beta&t=GxsLjCFo9AmdCTHkDLK622NOIF3hplTcipdZBeJbzPY)

In [4]:
from datetime import datetime
import pandas as pd

In [5]:
aws_cond_df = spark.createDataFrame(pd.read_csv('work/conditions.csv'))
aws_obs_df = spark.createDataFrame(pd.read_csv('work/observations.csv'))
aws_pats_df = spark.createDataFrame(pd.read_csv('work/patients.csv'))





In [9]:
aws_namespace = "test_aws.cancer"

spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {aws_namespace}")


DataFrame[]

In [14]:
spark.sql("SHOW namespaces in test_aws").show()

+---------------+
|      namespace|
+---------------+
|         cancer|
|acceptancetests|
|    namespaceph|
+---------------+



In [15]:



#
# AWS
#
# discriminator = datetime.now().strftime("%d%B%Y_%H_%M")
aws_namespace = "test_aws.cancer"
aws_tables = [(aws_cond_df,"conditions"),
              (aws_obs_df,"observations"),
              (aws_pats_df,"patients")]

# spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {aws_namespace}")
# print(f"Created namespace {aws_namespace}")

for df,t in aws_tables:
    aws_fqn = f"{aws_namespace}.{t}" 
    df.writeTo(aws_fqn).create()
    print(f"Wrote {aws_fqn} to AWS")

# spark.sql(f"SELECT * FROM {aws_fqn}").show()


Wrote test_aws.cancer.conditions to AWS
Wrote test_aws.cancer.observations to AWS
Wrote test_aws.cancer.patients to AWS
